<a href="https://colab.research.google.com/github/Faizan-ali16/Lunar-lander-AI-training-model/blob/main/lunar_lander_ppo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install swig
!pip -q install "gymnasium[box2d]" stable-baselines3 imageio imageio-ffmpeg

In [3]:
import time, numpy as np, gymnasium as gym, imageio.v2 as imageio, matplotlib
from functools import lru_cache
from gymnasium import Wrapper
from PIL import Image, ImageDraw, ImageFont
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import BaseCallback

ENV_ID = "LunarLander-v3"
N_ENVS = 16
MAX_STEPS = 2_000_000      # safety cap
MAX_MINUTES = 40           # safety cap
SOLVED = 240               # stops itself once the raw game average reaches this
MILESTONES = [0, 100, 300, 1000, 2500, 5000, 10000, 20000, 50000]
RESULTS = {}

# ---------- reward wrappers ----------
class TimePenalty(Wrapper):
    def step(self, a):
        o, r, te, tr, i = self.env.step(a)
        return o, r - 0.01, te, tr, i

class HoverHack(Wrapper):  # broken on purpose: +1 per frame for hovering
    def step(self, a):
        o, r, te, tr, i = self.env.step(a)
        if abs(o[4]) < 0.2 and abs(o[2]) < 0.3 and abs(o[3]) < 0.3:
            r += 1
        return o, r, te, tr, i

class HoverFix(Wrapper):  # pay for improvement, not for sitting in a nice spot
    def _phi(self, o):
        return -(abs(o[2]) + abs(o[3]) + abs(o[4]))
    def reset(self, **kw):
        o, i = self.env.reset(**kw)
        self.phi = self._phi(o)
        return o, i
    def step(self, a):
        o, r, te, tr, i = self.env.step(a)
        p = self._phi(o)
        r += 5 * (p - self.phi) - 0.01
        self.phi = p
        return o, r, te, tr, i

# ---------- video drawing ----------
W, H, X = 1280, 720, 912
BG, GREY = (11, 14, 22), (120, 130, 155)
FP = matplotlib.get_data_path() + "/fonts/ttf/DejaVuSans{}.ttf"

@lru_cache(None)
def F(size, bold=True):
    return ImageFont.truetype(FP.format("-Bold" if bold else ""), size)

def colr(v):
    return GREY if v is None else (255, 90, 90) if v < 0 else (255, 190, 60) if v < 200 else (80, 230, 140)

def make_panel(name, caption, count, avg, hist):
    im = Image.new("RGB", (W, H), BG)
    d = ImageDraw.Draw(im)
    d.text((24, 24), "LUNAR LANDER  |  PPO LEARNS TO LAND", font=F(28), fill=(235, 240, 255))
    d.text((W - 24, 30), name, font=F(20, False), fill=(130, 140, 165), anchor="ra")
    d.text((X, 100), caption, font=F(15), fill=GREY)
    d.text((X, 122), f"{count:,}", font=F(64), fill=(255, 255, 255))
    d.text((X, 212), "AVG SCORE, LAST 100 ATTEMPTS", font=F(15), fill=GREY)
    d.text((X, 234), "--" if avg is None else f"{avg:+.0f}", font=F(46), fill=colr(avg))
    x0, y0, x1, y1 = X, 330, W - 24, 500
    d.text((x0, y0 - 24), "SCORE vs ATTEMPTS (log scale)", font=F(13), fill=GREY)
    d.rectangle((x0, y0, x1, y1), outline=(45, 55, 80), width=2)
    ym = lambda v: y1 - (min(max(v, -400), 300) + 400) / 700 * (y1 - y0)
    xm = lambda e: x0 + np.log10(max(e, 1)) / 5 * (x1 - x0)
    for x in range(x0, x1, 12):
        d.line((x, ym(200), x + 6, ym(200)), fill=(60, 160, 110), width=2)
    d.text((x0 + 6, ym(200) + 4), "solved = 200", font=F(13, False), fill=(60, 160, 110))
    pts = [(xm(e), ym(v)) for e, v in hist]
    if len(pts) > 1:
        d.line(pts, fill=(90, 200, 255), width=3)
    if pts:
        px, py = pts[-1]
        d.ellipse((px - 6, py - 6, px + 6, py + 6), fill=(90, 200, 255))
    for i, t in enumerate(["1", "10", "100", "1k", "10k", "100k"]):
        d.text((x0 + i / 5 * (x1 - x0), y1 + 6), t, font=F(13, False), fill=GREY, anchor="ma")
    return im

def compose(base, frame, score, flight, status):
    im = base.copy()
    d = ImageDraw.Draw(im)
    im.paste(Image.fromarray(frame).resize((864, 576), Image.BICUBIC), (24, 96))
    d.text((X, 540), "THIS FLIGHT", font=F(15), fill=GREY)
    d.text((X, 562), f"{score:+.0f}", font=F(46), fill=colr(score))
    if status:
        c = {"LANDED": (60, 200, 120), "CRASHED": (230, 70, 70), "TIME OUT": (240, 180, 50)}[status]
        d.rounded_rectangle((X, 636, W - 24, 684), 10, fill=c)
        d.text(((X + W - 24) // 2, 660), status, font=F(26), fill=(10, 12, 18), anchor="mm")
    else:
        d.text((X, 650), f"flight {flight}", font=F(18, False), fill=GREY)
    return np.array(im)

def fade(img, n):
    return img if n >= 12 else (img * ((n + 1) / 12)).astype(np.uint8)

def record_clip(model, writer, name, caption, count, avg, hist, det, min_frames):
    base = make_panel(name, caption, count, avg, hist)
    env = gym.make(ENV_ID, render_mode="rgb_array")
    n, flight = 0, 0
    while n < min_frames and flight < 6:
        flight += 1
        obs, _ = env.reset()
        score, step, status = 0.0, 0, None
        while status is None:
            act, _ = model.predict(obs, deterministic=det)
            obs, r, term, trunc, _ = env.step(int(act))
            score += r
            step += 1
            status = ("LANDED" if r > 0 else "CRASHED") if term else ("TIME OUT" if trunc else None)
            if step <= 300 or step % 3 == 0 or status:
                img = compose(base, env.render(), score, flight, status)
                writer.append_data(fade(img, n))
                n += 1
        for _ in range(35):
            writer.append_data(img)
            n += 1
    env.close()

def outro(writer, mean, std, rate, steps, eps, why):
    im = Image.new("RGB", (W, H), BG)
    d = ImageDraw.Draw(im)
    c = W // 2
    d.text((c, 190), "FINAL TEST  |  reward tweaks off, 100 fresh flights", font=F(28), fill=GREY, anchor="mm")
    d.text((c, 320), f"{mean:+.0f}", font=F(150), fill=colr(mean), anchor="mm")
    d.text((c, 420), f"average score  (std {std:.0f})", font=F(30, False), fill=(235, 240, 255), anchor="mm")
    d.text((c, 490), f"{rate * 100:.0f}% of flights scored 200+", font=F(36), fill=(235, 240, 255), anchor="mm")
    d.text((c, 590), f"{eps:,} training attempts  |  {steps:,} steps  |  stopped: {why}", font=F(22, False), fill=GREY, anchor="mm")
    img = np.array(im)
    for k in range(150):
        writer.append_data(fade(img, k))

# ---------- training ----------
def evaluate(model, n):
    env = gym.make(ENV_ID)
    scores = []
    for i in range(n):
        obs, _ = env.reset(seed=10_000 + i)
        done, s = False, 0.0
        while not done:
            a, _ = model.predict(obs, deterministic=True)
            obs, r, te, tr, _ = env.step(int(a))
            s += r
            done = te or tr
        scores.append(s)
    env.close()
    sc = np.array(scores)
    return float(sc.mean()), float(sc.std()), float((sc >= 200).mean())

class Run(BaseCallback):
    def __init__(self, name, writer):
        super().__init__()
        self.name, self.writer = name, writer
        self.eps, self.hist, self.todo = 0, [], list(MILESTONES)
        self.rolls, self.solved, self.why, self.t0 = 0, False, "step limit", time.time()

    def avg(self):
        b = self.model.ep_info_buffer
        return float(np.mean([e["r"] for e in b])) if len(b) else None

    def _on_rollout_start(self):
        a = self.avg()
        if a is not None:
            self.hist.append((self.eps, a))
            print(f"{self.num_timesteps:>9,} steps | {self.eps:>6,} attempts | avg score {a:+7.1f}")
        if self.todo and self.eps >= self.todo[0]:
            while self.todo and self.eps >= self.todo[0]:
                self.todo.pop(0)
            cap = "FIRST ATTEMPT  |  UNTRAINED" if self.eps == 0 else "TRAINING ATTEMPTS SO FAR"
            record_clip(self.model, self.writer, self.name, cap, max(self.eps, 1), a, self.hist, False, 300)
        self.rolls += 1
        if self.rolls % 3 == 0:
            m, _, _ = evaluate(self.model, 20)
            print(f"          test on raw game (20 flights): {m:+.0f}")
            self.solved = m >= SOLVED

    def _on_step(self):
        self.eps += int(np.sum(self.locals["dones"]))
        if self.solved:
            self.why = "target reached"
            return False
        if time.time() - self.t0 > MAX_MINUTES * 60:
            self.why = "time limit"
            return False
        return True

def run(name, wrapper=None, max_steps=MAX_STEPS, seed=0):
    writer = imageio.get_writer(f"{name}.mp4", fps=50, codec="libx264", quality=8, pixelformat="yuv420p")
    try:
        env = make_vec_env(ENV_ID, n_envs=N_ENVS, seed=seed, wrapper_class=wrapper)
        model = PPO("MlpPolicy", env, n_steps=1024, batch_size=64, n_epochs=4, gamma=0.999,
                    gae_lambda=0.98, ent_coef=0.01, seed=seed, device="cpu", verbose=0)
        cb = Run(name, writer)
        model.learn(max_steps, callback=cb)
        a = cb.avg()
        hist = cb.hist + ([(cb.eps, a)] if a is not None else [])
        record_clip(model, writer, name, "FULLY TRAINED  |  ATTEMPTS", max(cb.eps, 1), a, hist, True, 450)
        mean, std, rate = evaluate(model, 100)
        outro(writer, mean, std, rate, model.num_timesteps, cb.eps, cb.why)
        model.save(f"{name}_model")
    finally:
        writer.close()
    RESULTS[name] = dict(score=round(mean, 1), std=round(std, 1), landed_pct=round(rate * 100), attempts=cb.eps, steps=model.num_timesteps, stopped=cb.why)
    print("\nDONE", name, RESULTS[name])
    return f"{name}.mp4"

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
video = run("1_time_penalty", TimePenalty)
from IPython.display import Video
Video(video, embed=True, width=900)

<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


   16,384 steps |    170 attempts | avg score  -172.8


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


   32,768 steps |    343 attempts | avg score  -151.1
          test on raw game (20 flights): -35
   49,152 steps |    514 attempts | avg score  -122.1
   65,536 steps |    683 attempts | avg score  -117.7
   81,920 steps |    845 attempts | avg score   -93.8
          test on raw game (20 flights): -181
   98,304 steps |    989 attempts | avg score   -77.2
  114,688 steps |  1,135 attempts | avg score   -70.3
  131,072 steps |  1,278 attempts | avg score   -48.5
          test on raw game (20 flights): -178
  147,456 steps |  1,400 attempts | avg score   -36.2
  163,840 steps |  1,520 attempts | avg score   -32.9
  180,224 steps |  1,611 attempts | avg score   -22.3
          test on raw game (20 flights): -158
  196,608 steps |  1,700 attempts | avg score    -6.0
  212,992 steps |  1,757 attempts | avg score    -1.3
  229,376 steps |  1,812 attempts | avg score    -3.1
          test on raw game (20 flights): -170
  245,760 steps |  1,844 attempts | avg score    -3.6
  262,144 steps

In [5]:
from google.colab import files
files.download("1_time_penalty.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
video = run("2_hover_hack", HoverHack, max_steps=1_000_000)
from IPython.display import Video
Video(video, embed=True, width=900)

   16,384 steps |    165 attempts | avg score  -194.8
   32,768 steps |    335 attempts | avg score  -171.3
          test on raw game (20 flights): -820
   49,152 steps |    503 attempts | avg score  -148.6
   65,536 steps |    664 attempts | avg score  -135.0
   81,920 steps |    829 attempts | avg score  -109.5
          test on raw game (20 flights): -834
   98,304 steps |    988 attempts | avg score   -97.5
  114,688 steps |  1,129 attempts | avg score   -70.1
  131,072 steps |  1,265 attempts | avg score   -52.2
          test on raw game (20 flights): -383
  147,456 steps |  1,384 attempts | avg score   -36.7
  163,840 steps |  1,484 attempts | avg score   -29.1
  180,224 steps |  1,553 attempts | avg score   -22.9
          test on raw game (20 flights): -661
  196,608 steps |  1,592 attempts | avg score   -27.2
  212,992 steps |  1,632 attempts | avg score   -60.8
  229,376 steps |  1,660 attempts | avg score   -82.1
          test on raw game (20 flights): -785
  245,760 step

In [7]:
import pandas as pd
display(pd.DataFrame(RESULTS).T)
from google.colab import files
for n in RESULTS: files.download(f"{n}.mp4")

,score,std,landed_pct,attempts,steps,stopped
1_time_penalty,250.5,21.3,100,2648,819216,target reached
2_hover_hack,-1.0,33.0,0,2510,1015808,step limit


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>